# 01 — Corpus Normalization

**Project:** Misurare la creatività linguistica: sorpresa ed entropia come strumenti di analisi stilistica  
**Prize:** Premio di ricerca Dino Buzzetti (AIUCD)  
**Author:** Silvia Lilli  
**Last updated:** 2026-06-07

---

## Description

This notebook implements the full preprocessing pipeline for the corpus of Italian literary prose texts:

1. **Encoding verification** — detect encoding of each file in `data/raw/`
2. **Encoding standardization** — convert all files to UTF-8, saved to `data/raw_utf8/`
3. **Graphic normalization** — apply text-specific normalization rules, saved to `data/normalized/`

Original files in `data/raw/` are never modified.
All decisions are documented in `docs/decisions_log.md`.

### Normalization rules applied

| File | Operation |
|------|-----------|
| `1878_dossi_desinenza_a.txt` | Removal of inverted punctuation (`¿`, `¡`); grave → acute accent on conjunctions, pronouns, numbers, non-standard oxytones, and passato remoto forms; non-standard accents on monosyllables and disyllables removed entirely; replacement of double comma `,,` with `,`; removal of accents on non-oxytone words (non-final position); replacement of semiconsonantal `j` with `i` (replacement of semiconsonantal `j` with `i`, excluding stoplist forms)|
| `1925_pirandello_uno_nessuno_centomila.txt` | Replace semiconsonantal `j` → `i` |
| `1869_tarchetti_fosca.txt` | Replace `í`→`ì`, `ú`→`ù` (acute → grave on `i` and `u`) |
| All other texts | No normalization — copied as-is |

## 0. Setup

Import libraries and define paths.

In [41]:
import os
import re
import shutil
import unicodedata
import chardet
import uuid

RAW_DIR      = "../data/raw"
RAW_UTF8_DIR = "../data/raw_utf8"
NORM_DIR     = "../data/normalized"

os.makedirs(RAW_UTF8_DIR, exist_ok=True)
os.makedirs(NORM_DIR, exist_ok=True)

print(f"Raw dir:        {os.path.abspath(RAW_DIR)}")
print(f"Raw UTF-8 dir:  {os.path.abspath(RAW_UTF8_DIR)}")
print(f"Normalized dir: {os.path.abspath(NORM_DIR)}")

Raw dir:        c:\Users\silvi\OneDrive\Università\PROGETTI\AIUCD Buzzetti\Project\surprisal-entropy-italian-literature\data\raw
Raw UTF-8 dir:  c:\Users\silvi\OneDrive\Università\PROGETTI\AIUCD Buzzetti\Project\surprisal-entropy-italian-literature\data\raw_utf8
Normalized dir: c:\Users\silvi\OneDrive\Università\PROGETTI\AIUCD Buzzetti\Project\surprisal-entropy-italian-literature\data\normalized


## 1. Helper functions

In [42]:
def read_text(path: str) -> str:
    """Read a UTF-8 text file."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def write_text(path: str, text: str) -> None:
    """Write a UTF-8 text file."""
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"  Saved → {os.path.basename(path)}")

## 2. Encoding verification

Detect the encoding of each file in `data/raw/` using `chardet`.
Results inform the conversion map in the next step.

In [43]:
print(f"{'File':<50} {'Encoding':<20} {'Confidence'}")
print("-" * 80)

for filename in sorted(f for f in os.listdir(RAW_DIR) if f.endswith(".txt")):
    path = os.path.join(RAW_DIR, filename)
    with open(path, "rb") as f:
        raw = f.read()
    result = chardet.detect(raw)
    confidence = f"{result['confidence']:.0%}"
    flag = " ⚠️  low confidence" if result['confidence'] < 0.70 else ""
    print(f"{filename:<50} {result['encoding']:<20} {confidence}{flag}")

File                                               Encoding             Confidence
--------------------------------------------------------------------------------
1869_tarchetti_fosca.txt                           utf-8                84%
1878_dossi_desinenza_a.txt                         Windows-1252         67% ⚠️  low confidence
1881_verga_malavoglia.txt                          utf-8                83%
1889_d_annunzio_piacere.txt                        utf-8                83%
1891_serao_paese_cuccagna.txt                      ISO-8859-1           61% ⚠️  low confidence
1895_fogazzaro_piccolo_mondo_antico.txt            utf-8                83%
1904_deledda_cenere.txt                            utf-8                84%
1919_tozzi_occhi_chiusi.txt                        ISO-8859-1           75%
1920_palazzeschi_il_codice_perela.txt              UTF-8-SIG            100%
1925_pirandello_uno_nessuno_centomila.txt          utf-8                85%
1926_svevo_coscienza_zeno.txt        

## 3. Encoding standardization

All files are converted to UTF-8 and saved to `data/raw_utf8/`.
Original files in `data/raw/` are not modified.

Encoding detected via `chardet`:

| File | Encoding | Note |
|------|----------|------|
| `1878_dossi_desinenza_a.txt` | Windows-1252 | confidence 67% — spot-check recommended |
| `1891_serao_paese_cuccagna.txt` | ISO-8859-1 | confidence 61% — spot-check recommended |
| `1919_tozzi_occhi_chiusi.txt` | ISO-8859-1 | confidence 75% |
| `1920_palazzeschi_il_codice_perela.txt` | UTF-8-SIG | BOM removed |
| All others | UTF-8 | no conversion needed |

> ⚠️ **Decision note:** Files with confidence below 70% require manual spot-check
> after conversion. See `docs/decisions_log.md`.

In [44]:
ENCODING_MAP = {
    "1878_dossi_desinenza_a.txt":            "windows-1252",
    "1891_serao_paese_cuccagna.txt":          "iso-8859-1",
    "1919_tozzi_occhi_chiusi.txt":            "iso-8859-1",
    "1920_palazzeschi_il_codice_perela.txt":  "utf-8-sig",
}

print("Standardizing encoding to UTF-8...\n")

for filename in sorted(f for f in os.listdir(RAW_DIR) if f.endswith(".txt")):
    src = os.path.join(RAW_DIR, filename)
    dst = os.path.join(RAW_UTF8_DIR, filename)
    encoding = ENCODING_MAP.get(filename, "utf-8")

    with open(src, "r", encoding=encoding) as f:
        text = f.read()
    with open(dst, "w", encoding="utf-8") as f:
        f.write(text)

    note = f"({encoding} → utf-8)" if encoding != "utf-8" else "(already utf-8)"
    print(f"  {note:35} → {filename}")

print("\n✓ All files written to data/raw_utf8/ as UTF-8.")

Standardizing encoding to UTF-8...

  (already utf-8)                     → 1869_tarchetti_fosca.txt
  (windows-1252 → utf-8)              → 1878_dossi_desinenza_a.txt
  (already utf-8)                     → 1881_verga_malavoglia.txt
  (already utf-8)                     → 1889_d_annunzio_piacere.txt
  (iso-8859-1 → utf-8)                → 1891_serao_paese_cuccagna.txt
  (already utf-8)                     → 1895_fogazzaro_piccolo_mondo_antico.txt
  (already utf-8)                     → 1904_deledda_cenere.txt
  (iso-8859-1 → utf-8)                → 1919_tozzi_occhi_chiusi.txt
  (utf-8-sig → utf-8)                 → 1920_palazzeschi_il_codice_perela.txt
  (already utf-8)                     → 1925_pirandello_uno_nessuno_centomila.txt
  (already utf-8)                     → 1926_svevo_coscienza_zeno.txt

✓ All files written to data/raw_utf8/ as UTF-8.


### 3a. Spot-check converted files

Visual verification of the first 300 characters for files with low encoding confidence.

In [45]:
SPOT_CHECK_FILES = [
    "1878_dossi_desinenza_a.txt",
    "1891_serao_paese_cuccagna.txt",
]

for filename in SPOT_CHECK_FILES:
    text = read_text(os.path.join(RAW_UTF8_DIR, filename))
    print(f"=== {filename} ===")
    print(text[:300])
    print()

=== 1878_dossi_desinenza_a.txt ===
Carlo Dossi

LA DESINENZA IN A 



Sezione di una casa civile a due piani.
      
      O Pùbblico, o solo mio Rè, si fà porta. Due lire e tu sei in teatro. ¡Ànimo! risparmia un pajo di guanti, un nastro, un fiore, un sacchettino di dolci, e ardisci di non scroccarmi il biglietto. ¿Chi è mai, che co

=== 1891_serao_paese_cuccagna.txt ===
Matilde Serao
Il paese di cuccagna

CAPITOLO I°

L'ESTRAZIONE DEL LOTTO

      
      Dopo mezzogiorno il sole penetrò nella piazzetta dei Banchi Nuovi, allargandosi dalla litografia Cardone alla farmacia Cappa e di là si venne allungando, risalendo tutta la strada di Santa Chiara, dando una insolit



## 4. Normalization functions

### 4a. Dossi — graphic normalization

Six normalization operations are applied to *La desinenza in A*:

1. **Inverted punctuation** — `¿` and `¡` are removed (the correct closing
   mark is already present in the text).

2. **Grave → acute accent** — Dossi systematically uses grave accent where
   Italian standard orthography requires acute. Affected categories:
   conjunctions and adverbs (*perché*, *poiché*, *benché* etc.), numbers
   (*ventitré*, *duecentotrentatré*), pronouns (*sé*), and passato remoto
   verb forms (*ripeté*, *ribatté*, *procedé*, *poté*, *credé*).
   Forms preserved with grave: *caffè*, *ahimè*, *canapè*, *cioè*, *mercè*,
   *ebbè* (interjection), *Mosè* (proper noun).

3. **Non-oxytone accent removal** — accents on non-final syllables are
   removed (proparoxytones and paroxytones). Oxytones are left untouched.
   Non-standard accents ono monosyllables and disyllables removed entirely (*me*, *te*, *re*, *oboe*, *aloe*, *fa*, *sta*, *va*, *sa*, *su*, *qui*, *qua*, *tu*, *so*, *mo*, *vo*,
   *do*, *sto*, *fu*, *pro*, *gli*, *blu*, *Po*).

> ⚠️ **Decision note:** The function targets words where the accent falls
> on a non-final syllable. Oxytones (accent on last syllable) are left
> untouched, as they follow standard Italian orthographic convention.
> See `docs/decisions_log.md` for full rationale.

> 📝 **Note:** Accent removal was applied throughout, including French and
> Latin loanwords and quotations. Orthographic accuracy in multilingual
> passages is not a priority: code-switching sections will already produce
> high surprisal values due to lexical and syntactic divergence from the
> Italian standard model.

4. **Semiconsonantal `j` → `i`** — applied with a stoplist exempting French,
   dialect, and proper noun forms: *je*, *j'*, *toujours*, *Jole*,
   *Injection-Brou*, *jaune*, *joues*, *scirpaije*, *bottije*, *voijo*,
   *Jesus*, *Majestad*, *Jockeyclub*, *Jezabel*.

5. **Contextual restoration** — `tè` (noun, beverage) restored in its single
   occurrence after global normalization: *"una tazza di quella tepida aqua
   che chiamano il tè"*.

6. **Double comma** — `,,` replaced with `,`.

In [51]:
def strip_accent_from_char(char: str) -> str:
    """Remove accent from a single accented character, returning the base letter."""
    normalized = unicodedata.normalize("NFD", char)
    return "".join(c for c in normalized if unicodedata.category(c) != "Mn")


def remove_non_oxytone_accents(text: str) -> str:
    """
    Remove accents from non-oxytone words (i.e., words where the accent
    does not fall on the final syllable).
    Strategy: tokenize by word boundary; for each word, if the accented vowel
    is NOT on the last character, strip the accent.
    """
    accented_vowels = re.compile(r"[àèéìíîòóùúûÀÈÉÌÍÎÒÓÙÚÛ]")

    def process_word(word: str) -> str:
        result = list(word)
        for i, ch in enumerate(result):
            if accented_vowels.match(ch):
                is_final = (i == len(result) - 1)
                if not is_final:
                    result[i] = strip_accent_from_char(ch)
        return "".join(result)

    tokens = re.split(r"(\W+)", text)
    return "".join(process_word(t) if re.match(r"\w", t) else t for t in tokens)


def normalize_inverted_punctuation(text: str) -> str:
    """Remove Spanish-style inverted punctuation marks (closing mark already present)."""
    return text.replace("¿", "").replace("¡", "")


# Words where grave accent should be replaced with acute
GRAVE_TO_ACUTE = {
    # Conjunctions and adverbs
    "nè": "né",
    "perchè": "perché",
    "chè": "ché",
    "finchè": "finché",
    "senonchè": "senonché",
    "poichè": "poiché",
    "benchè": "benché",
    "fuorchè": "fuorché",
    "quasichè": "quasiché",
    "giacchè": "giacché",
    "allorchè": "allorché",
    "perocchè": "perocché",
    "ecchè": "ecché",
    "salvochè": "salvoché",
    "purchè": "purché",
    "alcunchè": "alcunché",
    "forsechè": "forseché",
    # Numbers
    "ventitrè": "ventitré",
    "duecentotrentatrè": "duecentotrentatré",
    # Pronouns
    "sè": "sé",
    # Past tense (passato remoto)
    "ripetè": "ripeté",
    "ribattè": "ribatté",
    "procedè": "procedé",
    "potè": "poté",
    "credè": "credé",
}

# Words with non-standard accent to remove entirely
REMOVE_ACCENT_WORDS = {
    "oboè": "oboe",
    "aloè": "aloe",
    "mè": "me",
    "tè": "te",
    "rè": "re",
    "fà": "fa",
    "stà": "sta",
    "và": "va",
    "sà": "sa",
    "sù": "su",
    "quì": "qui",
    "quà": "qua",
    "tù": "tu",
    "sò": "so",
    "mò": "mo",
    "vò": "vo",
    "dò": "do",
    "stò": "sto",
    "fù": "fu",
    "prò": "pro",
    "glì": "gli",
    "blù": "blu",
    "pò": "Po",  # fiume Po — maiuscola preservata dalla logica esistente
}

# Words exempt from j→i replacement (French, dialect, proper nouns, other languages)
J_STOPLIST = {
    "je", "j'", "toujours", "jole", "injection-brou",
    "jaune", "joues", "scirpaije", "bottije", "voijo",
    "jesus", "majestad", "jockeyclub", "jezabel"
}

# Contextual restoration after global normalization
CONTEXTUAL_RESTORE = {
    "una tazza di quella tepida aqua che chiamano il te":
    "una tazza di quella tepida aqua che chiamano il tè"
}


def normalize_grave_to_acute(text: str) -> str:
    """
    Replace grave accent with acute on words where Dossi uses non-standard
    grave but Italian orthographic convention requires acute.
    Handles both lowercase and capitalized forms.
    """
    def replace_word(match):
        word = match.group(0)
        lower = word.lower()
        if lower in GRAVE_TO_ACUTE:
            replacement = GRAVE_TO_ACUTE[lower]
            if word[0].isupper():
                return replacement[0].upper() + replacement[1:]
            return replacement
        if lower in REMOVE_ACCENT_WORDS:
            replacement = REMOVE_ACCENT_WORDS[lower]
            if word[0].isupper():
                return replacement[0].upper() + replacement[1:]
            return replacement
        return word

    return re.sub(r'\b\w+[àèéìíîòóùúû]\b', replace_word, text)


def normalize_j(text: str) -> str:
    """Replace semiconsonantal j→i, preserving stoplist words."""
    placeholders = {}

    # Step 1: protect stoplist words with placeholders,
    # preserving original capitalization of each occurrence
    for word in J_STOPLIST:
        pattern = r'\b' + re.escape(word) + r'\b'
        matches = list(re.finditer(pattern, text, flags=re.IGNORECASE))
        # Process in reverse to preserve string positions
        for match in reversed(matches):
            original_form = match.group(0)
            placeholder = f"PLACEHOLDER_{uuid.uuid4().hex}"
            placeholders[placeholder] = original_form
            text = text[:match.start()] + placeholder + text[match.end():]

    # Step 2: global j→i replacement
    text = text.replace("j", "i").replace("J", "I")

    # Step 3: restore placeholders with original forms
    for placeholder, original_form in placeholders.items():
        text = text.replace(placeholder, original_form)

    return text


def apply_contextual_restore(text: str) -> str:
    """Restore forms that were incorrectly normalized (context-dependent)."""
    for wrong, correct in CONTEXTUAL_RESTORE.items():
        text = text.replace(wrong, correct)
    return text


def normalize_dossi(text: str) -> str:
    """Apply all Dossi-specific normalizations."""
    text = normalize_inverted_punctuation(text)
    text = normalize_grave_to_acute(text)
    text = remove_non_oxytone_accents(text)
    text = normalize_j(text)
    text = apply_contextual_restore(text)
    text = text.replace(",,", ",")
    return text

### 4b. Pirandello — semiconsonantal `j` → `i`

In *Uno, nessuno e centomila*, `j` is used as a semiconsonant (e.g., *jeri* for *ieri*).  
It is replaced with `i` throughout.

In [47]:
def normalize_pirandello(text: str) -> str:
    """Replace semiconsonantal j with i (both cases)."""
    return text.replace("j", "i").replace("J", "I")

### 4c. Tarchetti — acute accent → grave on `i` and `u`

In *Fosca*, `í` and `ú` (acute accent) are replaced with `ì` and `ù` (grave accent),
consistent with modern Italian orthographic convention.

In [48]:
def normalize_tarchetti(text: str) -> str:
    """Replace acute-accented i and u with grave-accented equivalents."""
    return text.replace("í", "ì").replace("ú", "ù").replace("Í", "Ì").replace("Ú", "Ù")

## 5. Normalization pipeline

Read from `data/raw_utf8/`, apply text-specific normalization rules,
save all files to `data/normalized/`.

In [52]:
NORMALIZATION_MAP = {
    "1878_dossi_desinenza_a.txt":                normalize_dossi,
    "1925_pirandello_uno_nessuno_centomila.txt":  normalize_pirandello,
    "1869_tarchetti_fosca.txt":                  normalize_tarchetti,
}

utf8_files = sorted(f for f in os.listdir(RAW_UTF8_DIR) if f.endswith(".txt"))
print(f"Found {len(utf8_files)} files in raw_utf8/:\n")

for filename in utf8_files:
    src = os.path.join(RAW_UTF8_DIR, filename)
    dst = os.path.join(NORM_DIR, filename)
    norm_fn = NORMALIZATION_MAP.get(filename)

    if norm_fn is None:
        shutil.copy2(src, dst)
        print(f"  Copied (no normalization)      → {filename}")
    else:
        text = read_text(src)
        normalized = norm_fn(text)
        write_text(dst, normalized)
        print(f"  Normalized [{norm_fn.__name__}] → {filename}")

print(f"\n✓ Done. {len(utf8_files)} files written to data/normalized/")

Found 11 files in raw_utf8/:

  Saved → 1869_tarchetti_fosca.txt
  Normalized [normalize_tarchetti] → 1869_tarchetti_fosca.txt
  Saved → 1878_dossi_desinenza_a.txt
  Normalized [normalize_dossi] → 1878_dossi_desinenza_a.txt
  Copied (no normalization)      → 1881_verga_malavoglia.txt
  Copied (no normalization)      → 1889_d_annunzio_piacere.txt
  Copied (no normalization)      → 1891_serao_paese_cuccagna.txt
  Copied (no normalization)      → 1895_fogazzaro_piccolo_mondo_antico.txt
  Copied (no normalization)      → 1904_deledda_cenere.txt
  Copied (no normalization)      → 1919_tozzi_occhi_chiusi.txt
  Copied (no normalization)      → 1920_palazzeschi_il_codice_perela.txt
  Saved → 1925_pirandello_uno_nessuno_centomila.txt
  Normalized [normalize_pirandello] → 1925_pirandello_uno_nessuno_centomila.txt
  Copied (no normalization)      → 1926_svevo_coscienza_zeno.txt

✓ Done. 11 files written to data/normalized/


## 6. Verification

Spot-check the normalization output on a sample of each modified text.
Compare raw_utf8 vs. normalized for the first 500 characters.

In [53]:
def spot_check(filename: str, n_chars: int = 500) -> None:
    """Print a comparison of raw_utf8 and normalized text."""
    raw  = read_text(os.path.join(RAW_UTF8_DIR, filename))
    norm = read_text(os.path.join(NORM_DIR, filename))
    print(f"=== {filename} ===")
    print("--- RAW_UTF8 ---")
    print(raw[:n_chars])
    print("--- NORMALIZED ---")
    print(norm[:n_chars])
    print()

for fname in NORMALIZATION_MAP:
    spot_check(fname)

=== 1878_dossi_desinenza_a.txt ===
--- RAW_UTF8 ---
Carlo Dossi

LA DESINENZA IN A 



Sezione di una casa civile a due piani.
      
      O Pùbblico, o solo mio Rè, si fà porta. Due lire e tu sei in teatro. ¡Ànimo! risparmia un pajo di guanti, un nastro, un fiore, un sacchettino di dolci, e ardisci di non scroccarmi il biglietto. ¿Chi è mai, che con un cinque-centèsimi in tasca, avrebbe tanta impudenza di domandare, per grazia, a un panattiere un panuccio? ¿non si paga, fors'anche, una sbornia che ti fà misurare la terra tra le fratellèvoli ris
--- NORMALIZED ---
Carlo Dossi

LA DESINENZA IN A 



Sezione di una casa civile a due piani.
      
      O Pubblico, o solo mio Re, si fa porta. Due lire e tu sei in teatro. Animo! risparmia un paio di guanti, un nastro, un fiore, un sacchettino di dolci, e ardisci di non scroccarmi il biglietto. Chi è mai, che con un cinque-centesimi in tasca, avrebbe tanta impudenza di domandare, per grazia, a un panattiere un panuccio? non si paga, fors'a